# Why Does Training Only on Correct Answers Look Like Both SFT and REINFORCE?

Suppose we give an LLM a programming problem, and it generates a correct solution:

```python
def square(x):
    return x * x
```

We want the model to become more likely to generate this solution in the future.

## 1. Supervised Fine-Tuning (SFT)

The most straightforward approach is to maximize the probability of the correct solution:

$$
\max_\theta \log \pi_\theta(y\mid x)
$$

Where:

- $x$ is the programming problem.
- $y$ is the complete solution.
- $\pi_\theta(y\mid x)$ is the model's probability of generating that solution.

Since a solution consists of multiple tokens:

$$
\log \pi_\theta(y\mid x)
=
\sum_t \log \pi_\theta(y_t\mid x,y_{<t})
$$

This is the standard SFT objective: increase the probability of the target tokens.

## 2. REINFORCE

Now consider the REINFORCE policy gradient:

$$
\nabla_\theta J
=
\mathbb E_{y\sim\pi_\theta}
\left[
R(y)\nabla_\theta\log\pi_\theta(y\mid x)
\right]
$$

Suppose we use binary rewards:

- Correct solution: $R(y)=1$
- Incorrect solution: $R(y)=0$

Then:

**For a correct solution:**

$$
R(y)\nabla_\theta\log\pi_\theta(y\mid x)
=
\nabla_\theta\log\pi_\theta(y\mid x)
$$

**For an incorrect solution:**

$$
R(y)\nabla_\theta\log\pi_\theta(y\mid x)=0
$$

Therefore, for a single successful trajectory:

$$
\boxed{
\nabla_\theta J_{\text{REINFORCE}}
=
-\nabla_\theta L_{\text{SFT}}
}
$$

The gradient directions are exactly the same!

However, this does **not** mean SFT and REINFORCE are identical algorithms.

The important difference is how trajectories are sampled, selected, and weighted during training.

## 3. The Key Difference: How Training Examples Are Weighted

Imagine two programming problems:

| | Easy Problem | Hard Problem |
|---|---|---|
| Model success rate | 90% | 1% |
| Sampled solutions | 100 | 100 |
| Correct solutions | ~90 | ~1 |

Let's compare three approaches.

### Approach A: Keep All Correct Solutions, Then SFT

We sample 100 solutions per problem, keep every correct solution, and perform SFT.

- Easy problem: ~90 training examples.
- Hard problem: ~1 training example.

The easy problem receives much more training weight simply because the model already solves it frequently.

### Approach B: Keep One Correct Solution per Problem, Then SFT

We sample until we find a correct solution for each problem, then keep one successful trajectory per problem.

- Easy problem: 1 training example.
- Hard problem: 1 training example.

Now both problems receive approximately equal training weight, provided we can find a correct solution for each.

### Approach C: REINFORCE

We sample 100 solutions per problem and use the reward-weighted policy gradient:

$$
\nabla_\theta J
\approx
\frac{1}{N}
\sum_{i=1}^{N}
R(y_i)\nabla_\theta\log\pi_\theta(y_i\mid x)
$$

For the easy problem, roughly 90 trajectories contribute nonzero gradients.

For the hard problem, roughly 1 trajectory contributes a nonzero gradient.

When averaging over all rollouts, the expected gradient for the hard problem is naturally weighted by its low success probability.

This leads to an important mathematical relationship.

For binary rewards:

$$
\boxed{
\nabla_\theta P_\theta(\text{success}\mid x)
=
P_\theta(\text{success}\mid x)
\cdot
\mathbb E_{y\sim\pi_\theta(\cdot\mid x,\text{success})}
\left[
\nabla_\theta\log\pi_\theta(y\mid x)
\right]
}
$$

In other words:

**The expected REINFORCE gradient equals the expected gradient from SFT on successful samples, multiplied by the model's current success probability.**

This identity holds when the successful samples come from the current policy and the gradients are evaluated at that same policy.

That is why training only on successful trajectories, often called **Rejection Sampling Fine-Tuning (RFT)**, is closely related to reinforcement learning, even though its optimization step uses an ordinary supervised loss.

## 4. What Happens When We Add GRPO?

Let's consider a more extreme example.

Suppose we have a difficult programming problem with a model success rate of just 1%.

We sample 100 solutions and finally find one correct solution.

The rewards are:

$$
R=[1,0,0,\ldots,0]
$$

All three methods see exactly the same data, but they use it differently.

### RFT: Learn From the Successful Solution

RFT says:

> We found a correct solution. Let's keep it and train the model to reproduce it.

It discards the 99 incorrect solutions and performs SFT on the successful one.

### REINFORCE: Weight Each Trajectory by Its Reward

REINFORCE says:

> The successful trajectory provides a positive gradient. The failed trajectories contribute zero under the simple binary-reward estimator.

The successful trajectory increases the probability of its sampled actions.

However, because success is rare, the expected policy gradient for this problem is still weighted by its low success probability.

### GRPO: Compare Solutions Within the Group

GRPO says:

> This solution succeeded while the other 99 failed. Therefore, it should receive a positive advantage, while the failures should receive negative advantages.

In simplified form:

$$
A_i=\frac{R_i-\bar R}{\sigma_R}
$$

For our 100 solutions:

$$
\bar R=0.01
$$

The successful solution receives a positive advantage, and the 99 failed solutions receive negative advantages.

GRPO therefore:

- Increases the probability of sampled actions in the successful trajectory.
- Decreases the probability of sampled actions in the unsuccessful trajectories.

Actual GRPO implementations also include mechanisms such as importance ratios, clipping, and potentially KL regularization.

## 5. The Big Picture

| Method | What It Learns From | How It Updates the Policy |
|---|---|---|
| **RFT** | Successful solutions only | SFT on filtered trajectories |
| **REINFORCE** | Reward of each sampled solution | Reward-weighted policy gradient |
| **GRPO** | Relative rewards within a group | Advantage-weighted policy gradient |

The essential difference is not simply whether the model learns from correct answers.

All three methods can increase the probability of a correct solution.

**The fundamental difference is how they assign training weight to successful and unsuccessful trajectories.**

In the 1%-success example:

- **RFT:** We found a correct answer. Learn from it.
- **REINFORCE:** This is a rare successful trajectory. It contributes a positive gradient, but the expected update is still influenced by the low success probability.
- **GRPO:** This answer is better than the other 99. Give it positive advantage and give the failures negative advantage.

**All three algorithms observe the same data, but they assign different learning weights to that data.**